[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlatcl/multilogue/blob/main/notebooks/00_access.ipynb)

# Access — units of civic text

### [Neil D. Lawrence](http://inverseprobability.com), University of Cambridge

### 2026-10-18

**Abstract**: Multilogue organises public-dialogue analysis around Access → Assess → Address.
Access is getting dialogue into computational form: loaders, units of text, and
provenance. This notebook teaches the *unit* contract and shows how named public
sources switch without changing the rest of the Address stack.


In [ ]:
#@title Setup (run first)
# Infrastructure only: install or locate multilogue, prepare figure directory.
# Scientific imports live in later cells.

import sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "matplotlib", "pandas", "numpy", "scikit-learn", "pyyaml", "pot",
        "git+https://github.com/mlatcl/multilogue.git",
    ])
else:
    try:
        import multilogue  # noqa: F401
    except ImportError:
        _here = pathlib.Path.cwd().resolve()
        for _cand in (_here, _here.parent, *_here.parents):
            if (_cand / "pyproject.toml").is_file() and (_cand / "multilogue").is_dir():
                if str(_cand) not in sys.path:
                    sys.path.insert(0, str(_cand))
                break

from multilogue.notebook import setup

FIG = setup(figures_dir="outputs/notebook_figures")
print("Figures →", FIG)


# Introduction

Public dialogue studies share a methodological spine even when corpora and
headline questions differ. Multilogue is that spine as a library. Consumers bring
Access loaders; the package expects a small, corpus-agnostic object — the
**unit** — so extraction, clustering, and weighting never learn the file format.

Access is deliberately thin. It does not clean for a research question, propose
topics, or call a language model. Those belong later. The only job here is to
make text electronically usable under a clear contract, with citation and licence
metadata intact.


## The unit

A unit is one analysable span. Required fields are an identifier, a document id,
and text. Optional fields carry context for quote checks and free-form metadata
(group labels, format tags, gold codes). Downstream stages should treat
everything beyond the three required fields as opaque until Address asks for it.

Why this shape? A consultation CSV, a Polis export, and a CIMT TSV must feed
the same Address path. The reusable method lives *above* the file format.


In [ ]:
from IPython.display import display
from multilogue.access import list_sources, load
from multilogue.protocols import Unit


In [ ]:
print("Registered sources:", list_sources())
bundle = load("cimt", process="Cycling-Dialogue-Bonn", max_units=20)
units = bundle["units"]
print(f"{len(units)} units; license={bundle['license']!r}")
display(units[0])


The first unit above is a dictionary conforming to the contract. Pedagogy uses
**public** open corpora (CIMT by default). Downloads land in a gitignored cache
(`MULTILOGUE_DATAPATH` or `~/.cache/multilogue`), never in the repository.


## Switching sources

Following the old `pods.datasets.<name>()` pattern, each public corpus is a named
loader. `load(name, **kwargs)` returns units plus citation and licence metadata.


In [ ]:
import pandas as pd

# Default public corpus: CIMT Bonn cycling dialogue. Polis is another CC BY option.
bundle = load("cimt", process="Cycling-Dialogue-Bonn", max_units=80)
# bundle = load("polis", conversation="vtaiwan.uberx", max_units=80)

units = bundle["units"]
print(bundle["citation"][:220], "…")
print("cache path:", bundle["path"])
df = pd.DataFrame(units)
display(df[["unit_id", "doc_id", "text"]].head())
print("docs:", df.doc_id.nunique(), "units:", len(df))


## Corpus shape (still Access)

Even before Assess, we can ask whether the switch left a usable shape — how many
documents per group, how long the texts are — without posing a research question.


In [ ]:
from multilogue.assess_base import unit_size_table, docs_per_group
from multilogue import viz


In [ ]:
sizes = unit_size_table(units)
docs = docs_per_group(units, "group") if any("group" in u for u in units) else None
display(sizes.describe())
if docs is not None:
    display(docs)
    viz.docs_per_group_bars(docs, path=FIG / "00_docs_per_group.png")
viz.unit_size_hist(sizes, group_col="group", units=units, path=FIG / "00_unit_sizes.png")


# Outlook

Access ends when units exist with provenance. The next notebook, Assess, asks
what you can say about those units *without* knowing the research question —
sizes, emptiness, crude vocabulary — so that quality work remains reusable when
the question changes.
